<!-- - Removed the timestamp attribute and retained extracted temporal information.
- Created cyclic time features (hour_sin, hour_cos) to represent the periodic nature of time.
- Generated 5-minute rolling mean features for heart rate, stress, and respiration rate to capture short-term physiological trends.
- Generated 5-minute rolling standard deviation features to capture physiological variability.
- Created lag features (lag1) to incorporate information from the previous minute.
- Generated change features (hr_change, stress_change) to capture sudden physiological fluctuations.
- Applied one-hot encoding to activity type to convert categorical activities into numerical features.
- Applied one-hot encoding to sleep level to represent sleep states numerically.
- Removed rows with missing values introduced by rolling-window and lag operations.
- Performed correlation analysis to evaluate the relationship between engineered features and cognitive activity. -->

1) Temporal features 
2) Sensor time-domain features
3) Interaction features
4) Pattern/window features
5) Frequency-domain features
6) Physiological ratios
7) Transition features
8) Composite features
9) Cleaning (dropna)


detailed description:
-  removeed the timestamp attribute and retained extracted temporal info
- craeted cyclic time features (hour_sin, hourcos, day_of_w sin/cosin encoding) to represent periodic temporal patterns
- created 5 min roll mean feature for heart rate, stress and respiration rate to capture short-term physiological trends
- created 5 min rolling std features to capture physiological variability
- created lag features to incorporate info from the prev min
- created change features (hr_change, stress_change) to capture  any sudden changes in  physiologivcal activities
- introduced physiological stability features to obtain short term variability in heart rate 
- engineered interaction features between physiological signals ( like heart rate*stress or even heart rate:respiration ratio) in order capture cross sensor dependencies.
- extracted freq domain features using roll window energy and FFT based power to capture periodic structure in physio signals.
- applied one-hot encoding to activity type to convert categorical activities into numerical features.
- applied one-hot encoding to sleep level to represent sleep states numerically.
- Final cleaning: removed all rows that contain at least 1 NaN val from df which might have been introduced by rolling win, lag, and freq domain feature engineering.
- Performed corelation analysis to oberve relationships b/w engineered features and cognitive activity.
- Inspectin of engineered features

# Feature Engineering

In [11]:
#Load Preprocessed Dataset
import pandas as pd
import numpy as np

df = pd.read_csv("../final_dfs/full_df_preprocessed.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

1) Temporal features

Cyclical encoding (time awareness)

In [12]:
# Create Temporal Features
df["hour"] = df["timestamp"].dt.hour

df["hour_sin"] = np.sin( 2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos( 2 * np.pi * df["hour"] / 24)

df["day_of_week"] = df["timestamp"].dt.dayofweek

# Cyclical encoding for day of week
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# Drop raw temporal features
df.drop(columns=["hour", "day_of_week"], inplace=True)

time domain sensory features

In [13]:
# rolling window : lambda = 5 mins 
WINDOW = 5

for col in ["heart_rate", "stress", "respiration_rate"]:

    df[f"{col}_mean_5"] = df[col].rolling(WINDOW).mean()
    df[f"{col}_std_5"] = df[col].rolling(WINDOW).std() 

In [14]:
# lag features (memory effect)
df["hr_lag1"] = df["heart_rate"].shift(1)
df["stress_lag1"] = df["stress"].shift(1)
df["resp_lag1"] = df["respiration_rate"].shift(1)

In [15]:
# Change features (physiological dynamics)
df["hr_change"] = df["heart_rate"] - df["heart_rate"].shift(1)
df["stress_change"] = df["stress"] - df["stress"].shift(1)

In [16]:
# Stability feature
df["hr_stability_5"] = (
    df["heart_rate"].rolling(5).std()
    / (df["heart_rate"].rolling(5).mean() + 1e-6)
)

interaction features:

In [17]:
# Physiological interactions
df["hr_stress"] = df["heart_rate"] * df["stress"]

df["hr_resp_ratio"] = df["heart_rate"] / (df["respiration_rate"] + 1e-6)

df["stress_resp"] = df["stress"] * df["respiration_rate"]

In [18]:
# Cognitive load proxy (cross-signal imbalance)
df["hr_stress_gap"] = df["heart_rate"] - df["stress"]

Window based temporal context features capturing short-term behavioral and physiological patterns

In [19]:
# short-term behavioral context
df["cognitive_past_5"] = df["is_cognitive"].rolling(5).mean()

df["high_stress_past_5"] = (
    (df["stress"] > df["stress"].median())
    .rolling(5)
    .mean()
)

frequency features

In [ ]:
from scipy.fft import fft

# Signal energy (heart rate variability strength) - 30 min window
df["hr_energy_30"] = df["heart_rate"].rolling(30).apply(
    lambda x: np.sum((x - np.mean(x))**2),
    raw=True
)

# FFT power - 30 min window
def fft_power(x):
    return np.sum(np.abs(fft(x))**2)

df["hr_fft_power_30"] = df["heart_rate"].rolling(30).apply(fft_power, raw=True)

In [21]:
df["hr_irregularity_30"] = df["heart_rate"].rolling(30).apply(
    lambda x: np.mean(np.abs(np.diff(x))),
    raw=True
)

physiological ratios

In [22]:
df["stress_per_hr"] = df["stress"] / (df["heart_rate"] + 1e-6)

df["resp_per_stress"] = df["respiration_rate"] / (df["stress"] + 1e-6)

transition features

In [23]:
df["hr_acceleration"] = df["hr_change"].diff()
df["stress_acceleration"] = df["stress_change"].diff()

Composite features

In [24]:
df["physio_stability"] = (
    df[["heart_rate", "stress", "respiration_rate"]]
    .rolling(10)
    .std()
    .mean(axis=1)
)

df["cognitive_load_index"] = (
    df["stress"] * 0.4 +
    df["heart_rate"] * 0.3 +
    df["respiration_rate"] * 0.3
)

Final cleaning: removes all rows that contain at least 1 NaN val from df

In [25]:
df = df.dropna()

print("Final shape:", df.shape)
print("Missing values:", df.isna().sum().sum())

Final shape: (11491, 46)
Missing values: 0


In [ ]:
# df.to_csv(
#     "../final_dfs/full_df_features.csv",
#     index=False
# )

# print("Feature engineered dataset saved.")

Inspection

In [26]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Shape: (11491, 46)

Columns:
['timestamp', 'stress', 'body_battery', 'respiration_rate', 'heart_rate', 'sleep_score', 'is_cognitive', 'activity_type_generic', 'activity_type_running', 'activity_type_sedentary', 'activity_type_unknown', 'activity_type_walking', 'sleep_level_deep', 'sleep_level_light', 'sleep_level_rem', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'heart_rate_mean_5', 'heart_rate_std_5', 'stress_mean_5', 'stress_std_5', 'respiration_rate_mean_5', 'respiration_rate_std_5', 'hr_lag1', 'stress_lag1', 'resp_lag1', 'hr_change', 'stress_change', 'hr_stability_5', 'hr_stress', 'hr_resp_ratio', 'stress_resp', 'hr_stress_gap', 'cognitive_past_5', 'high_stress_past_5', 'hr_energy_30', 'hr_fft_power_30', 'hr_irregularity_30', 'stress_per_hr', 'resp_per_stress', 'hr_acceleration', 'stress_acceleration', 'physio_stability', 'cognitive_load_index']

Data Types:
timestamp                  datetime64[ns]
stress                            float64
body_battery                      float

In [27]:
missing = df.isna().sum().sort_values(ascending=False)

print("\nMissing values per column:")
print(missing[missing > 0])

print("\nTotal missing values:", df.isna().sum().sum())


Missing values per column:
Series([], dtype: int64)

Total missing values: 0


In [28]:
print("\nCognitive class distribution:")
print(df["is_cognitive"].value_counts())

print("\nProportions:")
print(df["is_cognitive"].value_counts(normalize=True))


Cognitive class distribution:
is_cognitive
0    9567
1    1924
Name: count, dtype: int64

Proportions:
is_cognitive
0    0.832565
1    0.167435
Name: proportion, dtype: float64


In [29]:
sensor_features = [
    "heart_rate", "stress", "respiration_rate", "body_battery"
]

rolling_features = [col for col in df.columns if "mean" in col or "std" in col]

lag_features = [col for col in df.columns if "lag" in col]

change_features = [col for col in df.columns if "change" in col]

time_features = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos"
]

interaction_features = [
    col for col in df.columns
    if "*" in col or "ratio" in col or "interaction" in col or "gap" in col
]

print("Sensor:", sensor_features)
print("Rolling:", rolling_features)
print("Lag:", lag_features)
print("Change:", change_features)
print("Time:", time_features)
print("Interaction:", interaction_features)

Sensor: ['heart_rate', 'stress', 'respiration_rate', 'body_battery']
Rolling: ['heart_rate_mean_5', 'heart_rate_std_5', 'stress_mean_5', 'stress_std_5', 'respiration_rate_mean_5', 'respiration_rate_std_5']
Lag: ['hr_lag1', 'stress_lag1', 'resp_lag1']
Change: ['hr_change', 'stress_change']
Time: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
Interaction: ['respiration_rate', 'respiration_rate_mean_5', 'respiration_rate_std_5', 'hr_resp_ratio', 'hr_stress_gap', 'hr_acceleration', 'stress_acceleration']


In [30]:
corr = df.corr(numeric_only=True)

target_corr = corr["is_cognitive"].sort_values(ascending=False)

print("\nTop positive correlations:")
print(target_corr.head(15))

print("\nTop negative correlations:")
print(target_corr.tail(15))


Top positive correlations:
is_cognitive              1.000000
cognitive_past_5          0.971164
high_stress_past_5        0.406686
respiration_rate_std_5    0.310987
activity_type_generic     0.279255
stress_per_hr             0.230370
stress_mean_5             0.193147
stress                    0.178060
stress_lag1               0.170336
stress_resp               0.162890
body_battery              0.131169
hr_stress                 0.122737
cognitive_load_index      0.113270
hr_irregularity_30        0.058467
dow_sin                   0.056516
Name: is_cognitive, dtype: float64

Top negative correlations:
hr_lag1                   -0.060818
respiration_rate          -0.063227
resp_lag1                 -0.066380
dow_cos                   -0.066942
heart_rate                -0.067547
activity_type_walking     -0.077232
respiration_rate_mean_5   -0.084330
sleep_level_deep          -0.109988
sleep_level_rem           -0.113084
activity_type_sedentary   -0.117887
sleep_level_light       

In [31]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
timestamp,11491,2026-06-07 00:14:00,2026-06-03 00:29:00,2026-06-05 00:21:30,2026-06-07 00:14:00,2026-06-09 00:06:30,2026-06-10 23:59:00,NaN
stress,11491.0,22.014185,-2.0,6.0,17.0,31.0,99.0,22.562497
body_battery,11491.0,59.736011,21.0,41.0,59.0,78.0,100.0,21.909001
respiration_rate,11491.0,14.234314,5.0,12.9,14.36,15.72,23.66,2.270699
heart_rate,11491.0,69.021856,31.5,57.0,65.0,77.0,172.0,15.906258
sleep_score,11491.0,76.265251,63.0,73.0,79.0,80.0,90.0,7.564975
is_cognitive,11491.0,0.167435,0.0,0.0,0.0,0.0,1.0,0.37338
hour_sin,11491.0,0.0,-1.0,-0.707107,0.0,0.707107,1.0,0.708029
hour_cos,11491.0,-0.002524,-1.0,-0.707107,-0.0,0.707107,1.0,0.70624
dow_sin,11491.0,0.119713,-0.974928,-0.781831,0.0,0.781831,0.974928,0.735549


In [32]:
print("Final rows:", len(df))
print("Final columns:", len(df.columns))

print("\nAny infinite values?")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

Final rows: 11491
Final columns: 46

Any infinite values?
0
